In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab09.ipynb")

# Lab 9: Trees, Forests, and Neural Networks

Welcome to the ninth Data 102 lab!

The goal of this lab is to build intuition for the deep models we have been studying in lecture. You will compare a decision tree to a random forest to see the bias-variance tradeoff in action, see how adding more (random) hidden features changes the accuracy of a model, and implement backpropagation for a tiny neural network.

The code and responses you need to fill in are represented by `...`. There is additional documentation for each part as you go along.

In preparation for this lab, we recommend that you review the slides and demos from Lectures 20, 21, and 22.

## Collaboration Policy
Data science is a collaborative activity. While you may talk with others about the labs, we ask that you **write your solutions individually**.

## Completing the Lab

Labs are **not graded** this semester. They are provided as practice to help you engage with and reinforce the course material.

We strongly encourage you to work through the entire lab genuinely and use the provided tests to check your understanding. If you need help, you can attend lab-specific office hours on Mondays, tutoring, or other course office hours.

After completing the lab, we recommend navigating to **Kernel > Restart Kernel and Run All Cells** and ensuring that your code runs without errors. Restarting your kernel helps ensure that your code works from a clean state rather than relying on variables from previously executed cells.

Recommended completion by: **Wednesday, November 18th.**

`<Collaborator Name> <Collaborator e-mail>`

# Setup
Let's begin by importing the libraries we will use. You can find the documentation for the libraries here:
* matplotlib: https://matplotlib.org/3.1.1/contents.html
* numpy: https://docs.scipy.org/doc/
* pandas: https://pandas.pydata.org/pandas-docs/stable/
* seaborn: https://seaborn.pydata.org/
* scikit-learn: https://scikit-learn.org/stable/

In [ ]:
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None

%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns

from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

sns.set()

import hashlib
def get_hash(num, significance = 4):
    num = round(num, significance)
    """Helper function for assessing correctness"""
    return hashlib.md5(str(num).encode()).hexdigest()

# Question 1: Decision Trees vs. Random Forests

In this question, we'll work with the hybrid car dataset ([which you may remember from Data 8](https://inferentialthinking.com/chapters/15/1/Correlation.html)).

It contains data on 153 different models of hybrid car from 1997 to 2013, with the price (`msrp`), gas mileage (`mpg`), type of car (`class`), and how fast the car accelerates in km/hour/second (`acceleration`).

We're going to try to predict the price using other features of the car, and compare how a single decision tree and a random forest (a combination of many decision trees) behave.

In [ ]:
hybrid = pd.read_csv('hybrid.csv')

X_cols = ["year", "acceleration", "mpg"] # Columns used for prediction
y_col = "msrp" # The column we're trying to predict

hybrid

This cell generates all pairs of scatterplots for numerical variables in the data. You should see the same trends discussed in the chapter of the Data 8 textbook linked above.

In [ ]:
sns.pairplot(hybrid)

### 1(a) Splitting the data

We'll start by splitting the data into training and test sets. Use the scikit-learn function `train_test_split` to make two dataframes called `train` and `test`. The test set should have $30\%$ of the data (46 rows).

The `train_test_split` function has an argument called `random_state` that lets you ensure that it uses the same random split every time: you should set that argument to `101` to pass the tests.

In [ ]:
from sklearn.model_selection import train_test_split

train, test = ...
train, test = ...

In [ ]:
grader.check("q1a")

### 1(b) Predicting the output

#### 1.b.(i) Decision Tree

Recall that a decision tree is a method for classification and regression that uses a tree-like structure to decide what value to predict for a point.

In this question, we'll use a decision tree for regression instead of classification. When we built a decision tree for classification in lecture, we made decisions about splitting based on how homogeneous the $y$-values were. Now, we'll instead make splits based on the residuals for predicting at that node.

For example, if we make our first split based on whether or not `mpg <= M`, we'll have some average MSRP for the low-MPG cars and some average MSRP for the high-MPG cars. The split value `M` is chosen to minimize the squared error that remains after predicting each car's MSRP with the average of its group.

Use scikit-learn's `DecisionTreeRegressor` to predict the MSRP from year, acceleration, and MPG. Add a new column to the `train` and `test` dataframes called `tree_pred` with the predictions from the decision tree.

*Hint: throughout this question, you should use the default values of all parameters for all models we're experimenting with (except where we tell you otherwise).*

*Hint: for this lab, you don't need to worry about pandas warnings about setting a value on a copy of a slice.*

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor()

...

train["tree_pred"] = ...
test["tree_pred"] = ...

In [ ]:
grader.check("q1b_i")

Run the following cell that computes the training set error and test set error.

In [ ]:
train_rmse = np.mean((train["tree_pred"] - train["msrp"]) ** 2) ** 0.5
test_rmse = np.mean((test["tree_pred"] - test["msrp"]) ** 2) ** 0.5

print("Training set error for decision tree:", train_rmse)
print("Test set error for decision tree:    ", test_rmse)

#### 1.b.(ii) Random Forest

Recall that a random forest is the combination of a large number of decision trees, each trained on a bootstrap sample of the data and using a random subset of the features.

Compute the prediction from a random forest using scikit-learn's `RandomForestRegressor`, with the following parameters:
* 100 trees (default)
* no limit on each tree's depth (default)
* Use the `max_features` parameter to only use one feature for each tree. (*The recommended value for random forests is for each tree to only use 1/3 of the features, and in this case we have 3 features.*)

Add a new column to the `train` and `test` dataframes called `forest_pred` with the predictions from the random forest.

*Hint: your code should look very similar to your answer from 1.b.(i).*

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(max_features=1)

...

train["forest_pred"] = ...
test["forest_pred"] = ...

In [ ]:
grader.check("q1b_ii")

Run the following cell that computes the training set error and test set error.

In [ ]:
train_rmse = np.mean((train["forest_pred"] - train["msrp"]) ** 2) ** 0.5
test_rmse = np.mean((test["forest_pred"] - test["msrp"]) ** 2) ** 0.5

print("Training set error for random forest:", train_rmse)
print("Test set error for random forest:    ", test_rmse)

<!-- BEGIN QUESTION -->

### 1(c) Accuracy comparison

Of the decision tree and the random forest, which one does best on the training set? Why? Which model does best on the test set? Why? In your answer, mention whether each model has higher bias or higher variance.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 1(d) Looking inside the tree

Because a decision tree is a sequence of simple yes/no splits, we can inspect it. We'll use the `plot_tree` function to draw the decision tree:

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(16, 4))
plot_tree(tree_model);

We can see that the tree is quite deep and complex. Let's take a closer look at the nodes at the top:

In [ ]:
plt.figure(figsize=(12, 7))
plot_tree(tree_model, max_depth=2, fontsize=14, feature_names=X_cols);

There are a few things we can see right away:

* The first line tells us which feature to split on: values below the threshold go to the left, and values above go to the right.
* The third line tells us the number of training samples that made it that far into the tree.
* The fourth line tells us the average $y$-value (in this case, MSRP) of all the training samples that made it that far into the tree.

<!-- BEGIN QUESTION -->

Just by looking at the first few layers, we can already see that the decision tree has pulled out the most expensive cars into some of the branches, and the less expensive ones into other branches.

Suppose we had stopped growing the tree at this point. That would have given us four leaf nodes, each with very different mean MSRP. Describe the node that contains the most expensive cars in plain English. Then, explain in one or two sentences why a random forest is much harder to interpret in this way.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

# Question 2: Random Features as a Hidden Layer

In this question, we will see why neural networks can fit patterns that a linear model cannot, using the toy "ring" dataset. Let's start by loading the data, which has already been split for you into train and test sets:

In [ ]:
# This is the same plotting function from lecture

def draw_results(x1, x2, color, plot_title=''):
    plt.figure()
    plt.scatter(x1, x2, c=color, cmap='viridis', alpha=0.7);
    plt.colorbar()
    plt.title(plot_title)
    plt.axis('equal')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.tight_layout()

In [ ]:
# Import datasets
ring_train = pd.read_csv('ring_train.csv')
ring_test = pd.read_csv('ring_test.csv')
draw_results(ring_train['x1'], ring_train['x2'], color=ring_train['y'])

We know from lecture that without any additional features, logistic regression will use a line as a decision boundary.
Where would you draw the best line to classify these points? (No need to answer, but please think about it.)

We are now going to fit a simple logistic regression model on the data using the following lines of code.

In [ ]:
# No need to write any code here: just understand.
from sklearn.linear_model import LogisticRegression

X_train = ring_train[['x1', 'x2']].values
y_train = ring_train['y'].values

X_test = ring_test[['x1', 'x2']].values
y_test = ring_test['y'].values

model_simple_features = LogisticRegression(penalty=None, solver='lbfgs')
model_simple_features.fit(X_train, y_train)

probs = model_simple_features.predict_proba(X_test)[:, 1]
y_hat = (probs > 0.5).astype(np.int64)

draw_results(
    X_test[:, 0], X_test[:, 1], color=probs,
    plot_title="Logistic regression predicted probs (no feature eng)"
)
draw_results(
    X_test[:, 0], X_test[:, 1], color=y_hat,
    plot_title="Logistic regression prediction (no feature eng)"
)

accuracy = np.mean(y_hat == y_test)
print("Test accuracy without feature engineering:", accuracy)

Comparing the labels classified by the model with the true labels, we notice that the simple logistic regression model is not ideal, partly because the true decision boundary is nonlinear. In order to improve on the model, we will engineer new features.

To engineer new features, we'll take inspiration from neural networks, and add many random features, where each is a random linear combination of the inputs, where the coefficients will be random numbers between -1 and 1. Each random feature is exactly one *hidden unit* of a one-hidden-layer neural network whose first-layer weights are random instead of learned.

Don't forget that we also need to apply a nonlinearity, since otherwise a linear combination of linear combinations is still just a linear function of $x_1$ and $x_2$.

### 2(a) Add random features

Complete the cell below to add random features to the dataset. As described above, we first generate a pair of coefficients $(c_1, c_2)$ uniformly random from $(-1, 1)$ and then for both the training set and the test set, add an additional column whose values are $\sigma(c_1x_1 + c_2 x_2)$, where $\sigma$ is the sigmoid function.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def add_random_feature(train_data, test_data):
    # Returns the modified train_data and test_data
    coeffs = ...
    coeffs = ...
    # This code gives the feature a convenient name
    feat_name = f"σ({coeffs[0]:0.2f}x1 + {coeffs[1]:0.2f}x2)"

    for dataset in (train_data, test_data):
        linear_combination = np.dot(dataset[['x1', 'x2']], coeffs)
        feature = ...
        feature = ...
        dataset[feat_name] = feature
    return train_data, test_data

In [ ]:
grader.check("q2a")

Using the code you completed in 2(a), we can add any number of random features to both the training set and the test set. For example, here we add 10 random features and train a new logistic regression model on them.

In [ ]:
# This cell uses the code you wrote to add 10 random features to the
# train and test sets.
np.random.seed(102)

ring_train_feats = ring_train.copy()
ring_test_feats = ring_test.copy()
for i in range(10):
    ring_train_feats, ring_test_feats = (
        add_random_feature(ring_train_feats, ring_test_feats)
    )
ring_train_feats.head()

In [ ]:
# No need to write any code here: just understand.
X_train = ring_train_feats.iloc[:, 1:].values
y_train = ring_train_feats['y'].values

X_test = ring_test_feats.iloc[:, 1:].values
y_test = ring_test_feats['y'].values

model_features = LogisticRegression(penalty=None, solver='lbfgs')
model_features.fit(X_train, y_train)

probs = model_features.predict_proba(X_test)[:, 1]
y_hat = (probs > 0.5).astype(np.int64)

draw_results(
    X_test[:, 0], X_test[:, 1], color=probs,
    plot_title="Logistic regression predicted probs (random features)"
)

draw_results(
    X_test[:, 0], X_test[:, 1], color=y_hat,
    plot_title="Logistic regression prediction (random features)"
)

accuracy = np.mean(y_hat == y_test)
print("Test accuracy with 10 random features:", accuracy)

You should see that the accuracy is already much better. How does it change as we add more random features (i.e., as we make the hidden layer *wider*)?

### 2(b) Width vs. accuracy

Complete the function `test_accuracy_with_k_features(k, seed)` below. It should:
1. Set the numpy random seed to `seed`.
2. Make copies of `ring_train` and `ring_test`, and add `k` random features to the copies using `add_random_feature`.
3. Fit a logistic regression model (`LogisticRegression(penalty=None, solver='lbfgs')`) on **all** of the columns except `y`.
4. Return the accuracy of the model on the test set (the fraction of test points whose predicted label matches `y`).

*Hint: the code that adds 10 features and fits the model above is almost exactly what you need.*

In [ ]:
def test_accuracy_with_k_features(k, seed):
    ...
    ...

In [ ]:
grader.check("q2b")

Run the following cell to plot the test accuracy as a function of the number of random features, averaged over several random seeds. (The shaded band shows the minimum and maximum over the seeds.)

In [ ]:
ks = [0, 1, 2, 3, 5, 10, 25, 50, 100]
seeds = range(10)
results = np.array([[test_accuracy_with_k_features(k, s) for s in seeds] for k in ks])

plt.figure(figsize=(7, 4))
plt.plot(ks, results.mean(axis=1), marker='o')
plt.fill_between(ks, results.min(axis=1), results.max(axis=1), alpha=0.3)
plt.xscale('symlog')
plt.xlabel('Number of random features $k$')
plt.ylabel('Test accuracy')
plt.title('Test accuracy vs. number of random features');

<!-- BEGIN QUESTION -->

### 2(c) Interpretation

1. Describe the trend in the plot above. Roughly how many random features are needed before the model works well?
2. Why does adding random features help logistic regression, when the model is still just a linear function of the (new) features?
3. Why is the resulting model harder to interpret than the logistic regression with only $x_1$ and $x_2$? Why does the spread between the seeds shrink as $k$ grows?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

# Question 3: Backpropagation for a Wide ReLU Network

In this question, you will implement the forward and backward passes of a neural network with **one hidden layer of $H$ ReLU units**, and then see how the number of hidden units affects the function it can fit.

For a dataset with inputs $x_1,\dots,x_n \in \mathbb{R}$ (stored as an $n \times 1$ matrix $X$) and outputs $y_1, \dots, y_n$, the network computes

$$Z = X W_1 + b_1, \qquad A = \text{ReLU}(Z), \qquad \hat{y} = A W_2 + b_2,$$

where $W_1$ is $1 \times H$, $b_1$ is a vector of length $H$, $W_2$ is a vector of length $H$, $b_2$ is a scalar, and $\text{ReLU}$ is applied entrywise (so $Z$ and $A$ are $n \times H$ matrices). We train it with the mean squared error loss

$$L = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2.$$

In [ ]:
# Run this cell to create the data and a helper that stores the parameters of the network
rng = np.random.RandomState(0)
X_curve = np.linspace(-3, 3, 40).reshape(-1, 1)
y_curve = np.sin(X_curve[:, 0]) + rng.normal(0, 0.1, size=40)

def init_params(H, seed=0):
    r = np.random.RandomState(seed)
    return {"W1": r.normal(0, 1, size=(1, H)),
            "b1": r.normal(0, 1, size=H),
            "W2": r.normal(0, 1, size=H) / np.sqrt(H),
            "b2": 0.0}

### 3(a) Forward pass

Complete the function `forward(params, X, y)` so that it returns the tuple `(Z, A, y_hat, loss)` defined above. Here `params` is a dictionary with keys `"W1"`, `"b1"`, `"W2"`, `"b2"`, `X` is an $n \times 1$ array, and `y` is an array of length $n$.

*Hint: use `np.maximum(0, Z)` for the ReLU.*

In [ ]:
def forward(params, X, y):
    ...
    ...

In [ ]:
grader.check("q3a")

### 3(b) Backward pass

Complete the function `backward(params, X, y)` so that it returns a dictionary `grads` with keys `"W1"`, `"b1"`, `"W2"`, `"b2"` holding $\frac{\partial L}{\partial W_1}$, $\frac{\partial L}{\partial b_1}$, $\frac{\partial L}{\partial W_2}$, $\frac{\partial L}{\partial b_2}$ (each with the same shape as the corresponding parameter). Use the values from your `forward` function and work backwards through the network one step at a time:

1. $\frac{\partial L}{\partial \hat{y}}$ (a vector of length $n$),
2. $\frac{\partial L}{\partial W_2}$, $\frac{\partial L}{\partial b_2}$, and $\frac{\partial L}{\partial A}$ (an $n \times H$ matrix),
3. $\frac{\partial L}{\partial Z}$ (remember that the derivative of $\text{ReLU}(z)$ is $1$ if $z > 0$ and $0$ otherwise),
4. $\frac{\partial L}{\partial W_1}$ and $\frac{\partial L}{\partial b_1}$.

*Hint: it may help to first work out the case $n = 1$, $H = 1$, then think about which sums over the data points appear for $n > 1$.*

In [ ]:
def backward(params, X, y):
    Z, A, y_hat, loss = forward(params, X, y)
    n = len(y)
    ...
    ...

In [ ]:
grader.check("q3b")

Now we can train the network with gradient descent using your gradients. Run the following cell to fit the noisy sine curve with networks of different widths $H$.

In [ ]:
def train(H, steps=5000, lr=0.01, seed=0):
    params = init_params(H, seed)
    for _ in range(steps):
        grads = backward(params, X_curve, y_curve)
        for key in params:
            params[key] = params[key] - lr * grads[key]
    return params

X_grid = np.linspace(-3, 3, 300).reshape(-1, 1)
widths = [1, 2, 5, 50]
fig, axes = plt.subplots(1, len(widths), figsize=(16, 3.5), sharey=True)
for ax, H in zip(axes, widths):
    fitted = train(H)
    Z, A, y_hat_grid, _ = forward(fitted, X_grid, np.zeros(len(X_grid)))
    final_loss = forward(fitted, X_curve, y_curve)[-1]
    ax.scatter(X_curve[:, 0], y_curve, s=12)
    ax.plot(X_grid[:, 0], y_hat_grid, color='C1')
    ax.set_title(f"H = {H} (train MSE {final_loss:.3f})")
plt.tight_layout()

<!-- BEGIN QUESTION -->

### 3(c) Width and the fitted function

1. Looking at the plots above, describe how the fitted function changes as the number of hidden units $H$ increases. What shape does each fitted function have (think about what $\text{ReLU}$ units add up to), and why can a single unit not fit the curve?
2. In the backward pass, $\frac{\partial L}{\partial Z}$ is zero for every entry where $Z \le 0$. What does this mean for a hidden unit whose $Z$ is negative for *every* training point (a "dead" unit), and how might this hurt a narrow network more than a wide one?
3. Increasing $H$ always lowers the training error here. Does that mean the widest network is the best one? What would you do to decide, and what risk do you run if you pick $H$ using only the training error?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Congratulations! You have finished Lab 9!

You've reached the end of the lab! Make sure you double check your work and make sure that you've answered all the written portions of the lab. Labs are not graded, but we recommend restarting your kernel and running all cells to verify that your code runs from top to bottom.

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

In [ ]:
grader.export(pdf=False, force_save=True)